# MIMIC-Eye Preprocess

In [54]:
import pandas as pd
import numpy as np
import json

In [64]:
df = pd.read_csv("mimic_eye_survival_admissions.csv")
df.head()

,subject_id,hadm_id,admittime,endtime,duration_days,duration_hours,event,admission_type,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,patient_folder
0,10002428,20321825,2156-04-30 20:35:00,2156-05-03 16:36:00,2.834028,68.016667,0,EW EMER.,EMERGENCY ROOM,CHRONIC/LONG TERM ACUTE CARE,Medicare,ENGLISH,WIDOWED,WHITE,2156-04-30 18:30:00,2156-04-30 21:53:00,0,patient_10002428
1,10002428,23473524,2156-05-11 14:49:00,2156-05-22 14:16:00,10.977083,263.450000,0,EW EMER.,EMERGENCY ROOM,CHRONIC/LONG TERM ACUTE CARE,Medicare,ENGLISH,WIDOWED,WHITE,2156-05-11 11:29:00,2156-05-11 16:53:00,0,patient_10002428
2,10002428,25797028,2155-07-14 19:15:00,2155-07-15 18:37:00,0.973611,23.366667,0,EU OBSERVATION,EMERGENCY ROOM,NaN,Medicare,ENGLISH,WIDOWED,WHITE,2155-07-14 16:58:00,2155-07-14 20:04:00,0,patient_10002428
3,10002428,26549334,2160-07-15 23:37:00,2160-07-16 18:49:00,0.800000,19.200000,0,EU OBSERVATION,EMERGENCY ROOM,NaN,Medicare,ENGLISH,WIDOWED,WHITE,2160-07-15 17:34:00,2160-07-16 18:49:00,0,patient_10002428
4,10002428,28295257,2160-04-14 12:30:00,2160-04-18 16:00:00,4.145833,99.500000,0,OBSERVATION ADMIT,EMERGENCY ROOM,SKILLED NURSING FACILITY,Medicare,ENGLISH,WIDOWED,WHITE,2160-04-14 09:01:00,2160-04-14 14:28:00,0,patient_10002428


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18689 entries, 0 to 18688
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   subject_id            18689 non-null  int64  
 1   hadm_id               18689 non-null  int64  
 2   admittime             18689 non-null  object 
 3   endtime               18689 non-null  object 
 4   duration_days         18689 non-null  float64
 5   duration_hours        18689 non-null  float64
 6   event                 18689 non-null  int64  
 7   admission_type        18689 non-null  object 
 8   admission_location    18689 non-null  object 
 9   discharge_location    13925 non-null  object 
 10  insurance             18689 non-null  object 
 11  language              18689 non-null  object 
 12  marital_status        18572 non-null  object 
 13  race                  18689 non-null  object 
 14  edregtime             14624 non-null  object 
 15  edouttime          

In [65]:
duration_col = "duration_days"
event_col = "event"

drop_cols = [
        "admittime",
        "edregtime",
        "edouttime",
        "endtime",
        "duration_hours",
        "hospital_expire_flag",  # duplicate of event
        "discharge_location", # leaks data
        "patient_folder"]

drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

In [66]:
# Columns for embeddings (more categories, keep original granularity)
embed_cols = [
    "admission_type",
    "admission_location",
    "race",
]

embed_cols = [c for c in embed_cols if c in df.columns]

# Small-cardinality categoricals -> OHE
ohe_cols = [
    "insurance",
    "marital_status",
]

ohe_cols = [c for c in ohe_cols if c in df.columns]

In [67]:
 # Language: binary numeric: ENGLISH vs other
if "language" in df.columns:
    df["language_english"] = (df["language"] == "ENGLISH").astype("uint8")
else:
    df["language_english"] = 0

df.drop(columns=["language"], inplace=True)

# Fill missing for categorical columns
for col in embed_cols + ohe_cols:
    df[col] = df[col].fillna("UNKNOWN")

In [68]:
def assign_split(sid):
    if sid in train_subj:
        return "train"
    elif sid in val_subj:
        return "val"
    else:
        return "test"

In [69]:
rng = np.random.RandomState(42)
unique_subjects = df["subject_id"].unique()
rng.shuffle(unique_subjects)

n_subj = len(unique_subjects)
n_train = int(0.7 * n_subj)
n_val = int(0.15 * n_subj)

train_subj = set(unique_subjects[:n_train])
val_subj = set(unique_subjects[n_train:n_train + n_val])
test_subj = set(unique_subjects[n_train + n_val:])

df["split"] = df["subject_id"].apply(assign_split)

print(f"Total subjects: {n_subj}")
print(f"Train subjects: {len(train_subj)}")
print(f"Val subjects  : {len(val_subj)}")
print(f"Test subjects : {len(test_subj)}")

print("\nSplit distribution by rows:")
print(df["split"].value_counts())

Total subjects: 2908
Train subjects: 2035
Val subjects  : 436
Test subjects : 437

Split distribution by rows:
split
train    12933
test      3078
val       2678
Name: count, dtype: int64


In [70]:
df.head()

,subject_id,hadm_id,duration_days,event,admission_type,admission_location,insurance,marital_status,race,language_english,split
0,10002428,20321825,2.834028,0,EW EMER.,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,1,train
1,10002428,23473524,10.977083,0,EW EMER.,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,1,train
2,10002428,25797028,0.973611,0,EU OBSERVATION,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,1,train
3,10002428,26549334,0.800000,0,EU OBSERVATION,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,1,train
4,10002428,28295257,4.145833,0,OBSERVATION ADMIT,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,1,train


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18689 entries, 0 to 18688
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subject_id          18689 non-null  int64  
 1   hadm_id             18689 non-null  int64  
 2   duration_days       18689 non-null  float64
 3   event               18689 non-null  int64  
 4   admission_type      18689 non-null  object 
 5   admission_location  18689 non-null  object 
 6   insurance           18689 non-null  object 
 7   marital_status      18689 non-null  object 
 8   race                18689 non-null  object 
 9   language_english    18689 non-null  uint8  
 10  split               18689 non-null  object 
dtypes: float64(1), int64(3), object(6), uint8(1)
memory usage: 1.4+ MB


In [72]:
mappings = {}

for col in embed_cols:
    cats = sorted(df[col].unique())
    mapping = {c: i for i, c in enumerate(cats)}
    mappings[col] = {
        "mapping": mapping,
        "num_classes": len(cats),
    }
    df[f"{col}_id"] = df[col].map(mapping).astype("int64")  # safe int IDs

    print(f"\nColumn '{col}': {len(cats)} categories -> encoded as '{col}_id'")


Column 'admission_type': 9 categories -> encoded as 'admission_type_id'

Column 'admission_location': 11 categories -> encoded as 'admission_location_id'

Column 'race': 33 categories -> encoded as 'race_id'


In [73]:
 # Save mappings for model to build embedding layers
json_filename = "category_mappings.json"

with open(json_filename, "w") as f:
    json.dump(mappings, f, indent=2)

In [74]:
if ohe_cols:
    X_ohe = pd.get_dummies(df[ohe_cols], drop_first=False).astype("uint8")
    print(f"\nOHE columns expanded to: {X_ohe.shape[1]} features")
else:
    X_ohe = pd.DataFrame(index=df.index)


OHE columns expanded to: 8 features


In [75]:
id_cols = ["subject_id", "hadm_id"]
base_cols = [duration_col, event_col, "split", "language_english"]
embed_id_cols = [f"{c}_id" for c in embed_cols]

final_df = pd.concat(
    [
        df[id_cols + base_cols + embed_id_cols],
        X_ohe,
    ],
    axis=1,
)

print("\n--- FINAL DF INFO (EMBED VERSION) ---")
print(final_df.info())


--- FINAL DF INFO (EMBED VERSION) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18689 entries, 0 to 18688
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   subject_id               18689 non-null  int64  
 1   hadm_id                  18689 non-null  int64  
 2   duration_days            18689 non-null  float64
 3   event                    18689 non-null  int64  
 4   split                    18689 non-null  object 
 5   language_english         18689 non-null  uint8  
 6   admission_type_id        18689 non-null  int64  
 7   admission_location_id    18689 non-null  int64  
 8   race_id                  18689 non-null  int64  
 9   insurance_Medicaid       18689 non-null  uint8  
 10  insurance_Medicare       18689 non-null  uint8  
 11  insurance_Other          18689 non-null  uint8  
 12  marital_status_DIVORCED  18689 non-null  uint8  
 13  marital_status_MARRIED   18689 non-nu

In [76]:
final_df.head()

,subject_id,hadm_id,duration_days,event,split,language_english,admission_type_id,admission_location_id,race_id,insurance_Medicaid,insurance_Medicare,insurance_Other,marital_status_DIVORCED,marital_status_MARRIED,marital_status_SINGLE,marital_status_UNKNOWN,marital_status_WIDOWED
0,10002428,20321825,2.834028,0,train,1,5,2,28,0,1,0,0,0,0,0,1
1,10002428,23473524,10.977083,0,train,1,5,2,28,0,1,0,0,0,0,0,1
2,10002428,25797028,0.973611,0,train,1,4,2,28,0,1,0,0,0,0,0,1
3,10002428,26549334,0.800000,0,train,1,4,2,28,0,1,0,0,0,0,0,1
4,10002428,28295257,4.145833,0,train,1,6,2,28,0,1,0,0,0,0,0,1


# MNB

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("training_data.csv")

In [86]:
df.head()

,CONTRACT_ID,BORROWER_ID,CONTRACT_BANK_ID,CONTRACT_CREDIT_INTERMEDIARY,CONTRACT_CREDIT_LOSS,CONTRACT_CURRENCY,CONTRACT_DATE_OF_LOAN_AGREEMENT,CONTRACT_DEPT_SERVICE_TO_INCOME,CONTRACT_FREQUENCY_TYPE,CONTRACT_INCOME,...,CONTRACT_RISK_WEIGHTED_ASSETS,CONTRACT_TYPE_OF_INTEREST_REPAYMENT,BORROWER_BIRTH_YEAR,BORROWER_CITIZENSHIP,BORROWER_COUNTRY,BORROWER_COUNTY,BORROWER_TYPE_OF_CUSTOMER,BORROWER_TYPE_OF_SETTLEMENT,TARGET_EVENT,TARGET_EVENT_DAY
0,TpK8osXs,d8SqtuEV,1d42bbf5,2.0,0.0,31,2457052,NaN,479a2e13,NaN,...,1.00,NaN,1217.0,98.0,98.0,20.0,A,NaN,-,NaN
1,EtIEHrcH,lrdxML0g,1d42bbf5,NaN,0.0,31,2457036,NaN,479a2e13,NaN,...,74.17,NaN,NaN,NaN,NaN,NaN,A,NaN,-,NaN
2,1G10DfKj,gII7nnq4,1d42bbf5,2.0,16350.0,31,2457043,7.05,479a2e13,127305.0,...,74.77,100003.0,1199.0,98.0,98.0,179.0,A,7.0,-,NaN
3,2NLT774,MMkJ8z/e,1d42bbf5,NaN,0.0,31,2457038,NaN,479a2e13,NaN,...,0.99,NaN,1221.0,98.0,98.0,NaN,A,NaN,-,NaN
4,VpylRvay,M417onFP,1d42bbf5,2.0,2395.0,31,2457091,NaN,479a2e13,NaN,...,74.30,100002.0,1260.0,98.0,98.0,178.0,A,1.0,-,NaN


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1602753 entries, 0 to 1602752
Data columns (total 34 columns):
 #   Column                               Non-Null Count    Dtype  
---  ------                               --------------    -----  
 0   CONTRACT_ID                          1602753 non-null  object 
 1   BORROWER_ID                          1602753 non-null  object 
 2   CONTRACT_BANK_ID                     1602753 non-null  object 
 3   CONTRACT_CREDIT_INTERMEDIARY         1543331 non-null  float64
 4   CONTRACT_CREDIT_LOSS                 1566599 non-null  float64
 5   CONTRACT_CURRENCY                    1602753 non-null  int64  
 6   CONTRACT_DATE_OF_LOAN_AGREEMENT      1602753 non-null  int64  
 7   CONTRACT_DEPT_SERVICE_TO_INCOME      1401273 non-null  float64
 8   CONTRACT_FREQUENCY_TYPE              1602753 non-null  object 
 9   CONTRACT_INCOME                      1404731 non-null  float64
 10  CONTRACT_INSTALMENT_AMOUNT           288222 non-null   float64
 11

In [67]:
print(df.shape)
print(df.columns.tolist())

(1602753, 34)
['CONTRACT_ID', 'BORROWER_ID', 'CONTRACT_BANK_ID', 'CONTRACT_CREDIT_INTERMEDIARY', 'CONTRACT_CREDIT_LOSS', 'CONTRACT_CURRENCY', 'CONTRACT_DATE_OF_LOAN_AGREEMENT', 'CONTRACT_DEPT_SERVICE_TO_INCOME', 'CONTRACT_FREQUENCY_TYPE', 'CONTRACT_INCOME', 'CONTRACT_INSTALMENT_AMOUNT', 'CONTRACT_INSTALMENT_AMOUNT_2', 'CONTRACT_INTEREST_PERIOD', 'CONTRACT_INTEREST_RATE', 'CONTRACT_LGD', 'CONTRACT_LOAN_AMOUNT', 'CONTRACT_LOAN_CONTRACT_TYPE', 'CONTRACT_LOAN_TO_VALUE_RATIO', 'CONTRACT_LOAN_TYPE', 'CONTRACT_MARKET_VALUE', 'CONTRACT_MATURITY_DATE', 'CONTRACT_MORTGAGE_LENDING_VALUE', 'CONTRACT_MORTGAGE_TYPE', 'CONTRACT_REFINANCED', 'CONTRACT_RISK_WEIGHTED_ASSETS', 'CONTRACT_TYPE_OF_INTEREST_REPAYMENT', 'BORROWER_BIRTH_YEAR', 'BORROWER_CITIZENSHIP', 'BORROWER_COUNTRY', 'BORROWER_COUNTY', 'BORROWER_TYPE_OF_CUSTOMER', 'BORROWER_TYPE_OF_SETTLEMENT', 'TARGET_EVENT', 'TARGET_EVENT_DAY']


## Survival Analysis features

### Computing duration

In [5]:
# 1) Numeric day codes
df["CONTRACT_DATE_OF_LOAN_AGREEMENT"] = pd.to_numeric(
    df["CONTRACT_DATE_OF_LOAN_AGREEMENT"], errors="coerce"
)

df["TARGET_EVENT_DAY"] = pd.to_numeric(
    df["TARGET_EVENT_DAY"], errors="coerce"
)

# 2) Last observation day for censoring
last_observed_day = np.nanmax([
    df["TARGET_EVENT_DAY"].max(),
    df["CONTRACT_DATE_OF_LOAN_AGREEMENT"].max()
])

# 3) Clean TARGET_EVENT
df["TARGET_EVENT"] = df["TARGET_EVENT"].astype(str).str.strip()
df["TARGET_EVENT"] = df["TARGET_EVENT"].replace({"-": "", "nan": ""})

# 4) Event day vs censoring
has_event = df["TARGET_EVENT"].isin(["K", "E"])
df["event_day_effective"] = np.where(
    has_event,
    df["TARGET_EVENT_DAY"],
    last_observed_day
)

# 5) Time from contract start to event/censor
df["time_contract"] = df["event_day_effective"] - df["CONTRACT_DATE_OF_LOAN_AGREEMENT"]
df["time_contract"] = df["time_contract"].clip(lower=0)

### Asign event codes

event_contract ∈ {0, 1, 2}

0 -> censored (no event)

1 -> default (TARGET_EVENT == 'K')

2 -> prepayment (TARGET_EVENT == 'E')

==> competing risks survival

In [6]:
def map_event(x):
    if x == "K":
        return 1   # Default event
    elif x == "E":
        return 2   # Prepayment (competing event)
    else:
        return 0   # Censored / no event within window

df["event_contract"] = df["TARGET_EVENT"].apply(map_event)

In [7]:
df["event_contract"].value_counts()

,count
event_contract,
0,1548364
2,43515
1,10874


### Reduce to one row per contract

A contract may appear multiple times, because multiple borrowers belong to it

For survival analysis we need ONE survival record per CONTRACT, containing:
 - event at the contract level
 - time-to-event at the contract level
 - contract-level features from CONTRACT_* columns

 ==> collapse all rows belonging to the same CONTRACT_ID

CONTRACT_* features should not differ across borrowers,
but the competition description says sometimes they do (inconsistencies)

==> Sort rows by CONTRACT_ID and take the first occurrence

In [8]:
CONTRACT_ID_COL = "CONTRACT_ID"

# Contract feature columns EXCLUDING CONTRACT_ID itself
contract_feature_cols = [
    c for c in df.columns
    if c.startswith("CONTRACT_") and c != CONTRACT_ID_COL
]

# Contract-level columns = all CONTRACT_* plus the survival fields created
contract_level_cols = [CONTRACT_ID_COL, "time_contract", "event_contract"] + contract_feature_cols

# Reduce to 1 row per contract
contract_base = (
    df.sort_values(CONTRACT_ID_COL)
      .drop_duplicates(subset=[CONTRACT_ID_COL])
      [contract_level_cols]
      .reset_index(drop=True)
)

contract_base.columns

Index(['CONTRACT_ID', 'time_contract', 'event_contract', 'CONTRACT_BANK_ID',
       'CONTRACT_CREDIT_INTERMEDIARY', 'CONTRACT_CREDIT_LOSS',
       'CONTRACT_CURRENCY', 'CONTRACT_DATE_OF_LOAN_AGREEMENT',
       'CONTRACT_DEPT_SERVICE_TO_INCOME', 'CONTRACT_FREQUENCY_TYPE',
       'CONTRACT_INCOME', 'CONTRACT_INSTALMENT_AMOUNT',
       'CONTRACT_INSTALMENT_AMOUNT_2', 'CONTRACT_INTEREST_PERIOD',
       'CONTRACT_INTEREST_RATE', 'CONTRACT_LGD', 'CONTRACT_LOAN_AMOUNT',
       'CONTRACT_LOAN_CONTRACT_TYPE', 'CONTRACT_LOAN_TO_VALUE_RATIO',
       'CONTRACT_LOAN_TYPE', 'CONTRACT_MARKET_VALUE', 'CONTRACT_MATURITY_DATE',
       'CONTRACT_MORTGAGE_LENDING_VALUE', 'CONTRACT_MORTGAGE_TYPE',
       'CONTRACT_REFINANCED', 'CONTRACT_RISK_WEIGHTED_ASSETS',
       'CONTRACT_TYPE_OF_INTEREST_REPAYMENT'],
      dtype='object')

### Add borrower features at contract level

Attach one borrower row per contract (the “first” borrower) to contract_base

In [9]:
# Identify borrower columns
borrower_cols = [c for c in df.columns if c.startswith("BORROWER_")]

# Take first borrower per contract
borrower_first = (
    df.sort_values(CONTRACT_ID_COL)
      .drop_duplicates(subset=[CONTRACT_ID_COL])
      [[CONTRACT_ID_COL] + borrower_cols]
)

# Merge into contract_base
contract_full = contract_base.merge(
    borrower_first,
    on=CONTRACT_ID_COL,
    how="left"
)

contract_full.head()

,CONTRACT_ID,time_contract,event_contract,CONTRACT_BANK_ID,CONTRACT_CREDIT_INTERMEDIARY,CONTRACT_CREDIT_LOSS,CONTRACT_CURRENCY,CONTRACT_DATE_OF_LOAN_AGREEMENT,CONTRACT_DEPT_SERVICE_TO_INCOME,CONTRACT_FREQUENCY_TYPE,...,CONTRACT_REFINANCED,CONTRACT_RISK_WEIGHTED_ASSETS,CONTRACT_TYPE_OF_INTEREST_REPAYMENT,BORROWER_ID,BORROWER_BIRTH_YEAR,BORROWER_CITIZENSHIP,BORROWER_COUNTRY,BORROWER_COUNTY,BORROWER_TYPE_OF_CUSTOMER,BORROWER_TYPE_OF_SETTLEMENT
0,+++-xRe,920.0,0,1d42bbf5,2.0,0.0,31,2457198,NaN,479a2e13,...,2.0,0.98,NaN,k5uB2TBT,1209.0,98.0,98.0,168.0,A,NaN
1,+++OCMUD,138.0,0,caa130b5,1.0,275.0,31,2457980,17.39,479a2e13,...,2.0,47.13,100003.0,duH32pIZ,1256.0,98.0,98.0,54.0,A,4.0
2,+++TRno4,703.0,0,1d42bbf5,2.0,0.0,31,2457415,NaN,479a2e13,...,2.0,0.96,NaN,6STCdPgA,1246.0,98.0,98.0,190.0,A,NaN
3,++-9T-UX,351.0,0,caa130b5,2.0,0.0,31,2457767,22.49,479a2e13,...,2.0,26.24,100003.0,z6LIJzZy,1256.0,98.0,98.0,59.0,B,NaN
4,++-9hmVF,9.0,0,f789f8b0,2.0,3072.0,31,2458109,20.62,87db11f5,...,2.0,84.86,100003.0,JfUP8uk0,1234.0,98.0,98.0,136.0,A,7.0


## Prepare for DeepHit and DeepSurv

In [10]:
contract_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1274533 entries, 0 to 1274532
Data columns (total 34 columns):
 #   Column                               Non-Null Count    Dtype  
---  ------                               --------------    -----  
 0   CONTRACT_ID                          1274533 non-null  object 
 1   time_contract                        1274533 non-null  float64
 2   event_contract                       1274533 non-null  int64  
 3   CONTRACT_BANK_ID                     1274533 non-null  object 
 4   CONTRACT_CREDIT_INTERMEDIARY         1220538 non-null  float64
 5   CONTRACT_CREDIT_LOSS                 1242976 non-null  float64
 6   CONTRACT_CURRENCY                    1274533 non-null  int64  
 7   CONTRACT_DATE_OF_LOAN_AGREEMENT      1274533 non-null  int64  
 8   CONTRACT_DEPT_SERVICE_TO_INCOME      1084370 non-null  float64
 9   CONTRACT_FREQUENCY_TYPE              1274533 non-null  object 
 10  CONTRACT_INCOME                      1086147 non-null  float64
 11

### Drop columns with > 70% missing

Columns with extremely high missingness tend to add more noise than signal.
Drop any column where more than 70% of the values are missing.

In [11]:
# Compute missing fraction per column
MISSING_THRESHOLD = 0.7
missing_frac = contract_full.isna().mean()

cols_to_drop = missing_frac[missing_frac > MISSING_THRESHOLD].index.tolist()
print("Dropping columns with > 70% missing:")
print(cols_to_drop)

# Drop those columns
contract_full = contract_full.drop(columns=cols_to_drop)
drop_manual = ["CONTRACT_BANK_ID"]
contract_full = contract_full.drop(columns=drop_manual)

contract_full.info()

Dropping columns with > 70% missing:
['CONTRACT_INSTALMENT_AMOUNT', 'CONTRACT_LOAN_TO_VALUE_RATIO', 'CONTRACT_MARKET_VALUE', 'CONTRACT_MORTGAGE_LENDING_VALUE', 'CONTRACT_MORTGAGE_TYPE']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1274533 entries, 0 to 1274532
Data columns (total 28 columns):
 #   Column                               Non-Null Count    Dtype  
---  ------                               --------------    -----  
 0   CONTRACT_ID                          1274533 non-null  object 
 1   time_contract                        1274533 non-null  float64
 2   event_contract                       1274533 non-null  int64  
 3   CONTRACT_CREDIT_INTERMEDIARY         1220538 non-null  float64
 4   CONTRACT_CREDIT_LOSS                 1242976 non-null  float64
 5   CONTRACT_CURRENCY                    1274533 non-null  int64  
 6   CONTRACT_DATE_OF_LOAN_AGREEMENT      1274533 non-null  int64  
 7   CONTRACT_DEPT_SERVICE_TO_INCOME      1084370 non-null  float64
 8   CONTRACT_FREQUEN

### Create survival labels

Prepare:
- `time_contract` = time to event or censor (already computed)
- `event_contract` = 0 (censored), 1 (default), 2 (prepay)
- `event_binary`   = 1 if default, 0 otherwise (for DeepSurv)

In [12]:
# Binary event for DeepSurv: default (1) vs everything else (0)
contract_full["event_binary"] = (contract_full["event_contract"] == 1).astype(int)

In [13]:
# Quick check
contract_full["event_contract"].value_counts(), contract_full["event_binary"].value_counts()

(event_contract
 0    1230660
 2      33908
 1       9965
 Name: count, dtype: int64,
 event_binary
 0    1264568
 1       9965
 Name: count, dtype: int64)

### Define ID, target, and feature columns

Exclude:
- IDs: `CONTRACT_ID`, `BORROWER_ID`
- Targets: `time_contract`, `event_contract`, `event_binary`

Everything else becomes a candidate feature.

In [14]:
id_cols = ["CONTRACT_ID", "BORROWER_ID"]
target_cols = ["time_contract", "event_contract", "event_binary"]

feature_cols = [
    c for c in contract_full.columns
    if c not in id_cols + target_cols
]

print("Number of feature columns:", len(feature_cols))

Number of feature columns: 24


### Decide numeric vs categorical feature columns

Some columns are stored as integers but are actually **codes** (e.g. currency, loan type).
Explicitly mark these as categorical, and treat the rest of numeric columns as real-valued.

In [15]:
numeric_cols_raw = contract_full[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols_raw:
    nunique = contract_full[col].nunique()
    print(f"{col}: {nunique} unique values")

CONTRACT_CREDIT_INTERMEDIARY: 4 unique values
CONTRACT_CREDIT_LOSS: 165007 unique values
CONTRACT_CURRENCY: 3 unique values
CONTRACT_DATE_OF_LOAN_AGREEMENT: 1072 unique values
CONTRACT_DEPT_SERVICE_TO_INCOME: 11367 unique values
CONTRACT_INCOME: 410880 unique values
CONTRACT_INSTALMENT_AMOUNT_2: 158047 unique values
CONTRACT_INTEREST_PERIOD: 1038 unique values
CONTRACT_INTEREST_RATE: 4322 unique values
CONTRACT_LGD: 879303 unique values
CONTRACT_LOAN_AMOUNT: 835480 unique values
CONTRACT_LOAN_CONTRACT_TYPE: 8 unique values
CONTRACT_MATURITY_DATE: 10640 unique values
CONTRACT_REFINANCED: 4 unique values
CONTRACT_RISK_WEIGHTED_ASSETS: 23085 unique values
CONTRACT_TYPE_OF_INTEREST_REPAYMENT: 9 unique values
BORROWER_BIRTH_YEAR: 101 unique values
BORROWER_CITIZENSHIP: 41 unique values
BORROWER_COUNTRY: 53 unique values
BORROWER_COUNTY: 199 unique values
BORROWER_TYPE_OF_SETTLEMENT: 8 unique values


In [16]:
# Columns to force as categorical (codes / IDs / enums)
categorical_force = [
    # contract-side categoricals
    "CONTRACT_CURRENCY",
    "CONTRACT_FREQUENCY_TYPE",
    "CONTRACT_LOAN_CONTRACT_TYPE",
    "CONTRACT_LOAN_TYPE",
    "CONTRACT_TYPE_OF_INTEREST_REPAYMENT",
    "CONTRACT_CREDIT_INTERMEDIARY",
    "CONTRACT_REFINANCED",
    # borrower-side categoricals
    "BORROWER_CITIZENSHIP",
    "BORROWER_COUNTRY",
    "BORROWER_COUNTY",
    "BORROWER_TYPE_OF_CUSTOMER",
    "BORROWER_TYPE_OF_SETTLEMENT",
]

# keep only those that still exist
categorical_force = [c for c in categorical_force if c in contract_full.columns]

# Initial numeric columns by dtype
numeric_cols_raw = contract_full[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Final numeric cols = numeric dtype but NOT forced categorical
numeric_cols = [c for c in numeric_cols_raw if c not in categorical_force]

# Final categorical cols = all feature_cols that are not numeric
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print("Numeric columns:")
print(numeric_cols)
print("\nCategorical columns:")
print(categorical_cols)

Numeric columns:
['CONTRACT_CREDIT_LOSS', 'CONTRACT_DATE_OF_LOAN_AGREEMENT', 'CONTRACT_DEPT_SERVICE_TO_INCOME', 'CONTRACT_INCOME', 'CONTRACT_INSTALMENT_AMOUNT_2', 'CONTRACT_INTEREST_PERIOD', 'CONTRACT_INTEREST_RATE', 'CONTRACT_LGD', 'CONTRACT_LOAN_AMOUNT', 'CONTRACT_MATURITY_DATE', 'CONTRACT_RISK_WEIGHTED_ASSETS', 'BORROWER_BIRTH_YEAR']

Categorical columns:
['CONTRACT_CREDIT_INTERMEDIARY', 'CONTRACT_CURRENCY', 'CONTRACT_FREQUENCY_TYPE', 'CONTRACT_LOAN_CONTRACT_TYPE', 'CONTRACT_LOAN_TYPE', 'CONTRACT_REFINANCED', 'CONTRACT_TYPE_OF_INTEREST_REPAYMENT', 'BORROWER_CITIZENSHIP', 'BORROWER_COUNTRY', 'BORROWER_COUNTY', 'BORROWER_TYPE_OF_CUSTOMER', 'BORROWER_TYPE_OF_SETTLEMENT']


### Handle missing values

Impute:
- Numeric features with column median
- Categorical features with the string `"missing"`

In [17]:
# Numeric imputation with median
contract_full[numeric_cols] = contract_full[numeric_cols].fillna(
    contract_full[numeric_cols].median()
)

# Categorical imputation with a special category
contract_full[categorical_cols] = contract_full[categorical_cols].fillna("missing")
# Convert all categorical features to string
contract_full[categorical_cols] = contract_full[categorical_cols].astype(str)

# Sanity check: no NaNs in features
print("Any NaNs left in numeric features?", contract_full[numeric_cols].isna().any().any())
print("Any NaNs left in categorical features?", contract_full[categorical_cols].isna().any().any())

Any NaNs left in numeric features? False
Any NaNs left in categorical features? False


### Train / validation / test split

Split the contracts into:
- 70% train
- 15% validation
- 15% test

Stratify by `event_binary` (default vs non-default) so class proportions stay similar.

In [18]:
from sklearn.model_selection import train_test_split

X = contract_full[feature_cols]
y_time = contract_full["time_contract"].values
y_event_multi = contract_full["event_contract"].values
y_event_bin = contract_full["event_binary"].values

# Train vs temp (val+test)
X_train, X_temp, y_time_train, y_time_temp, \
y_event_multi_train, y_event_multi_temp, \
y_event_bin_train, y_event_bin_temp = train_test_split(
    X,
    y_time,
    y_event_multi,
    y_event_bin,
    test_size=0.3,
    random_state=42,
    stratify=y_event_bin
)

# Temp -> val & test (equal split of remaining 30% => 15%/15%)
X_val, X_test, y_time_val, y_time_test, \
y_event_multi_val, y_event_multi_test, \
y_event_bin_val, y_event_bin_test = train_test_split(
    X_temp,
    y_time_temp,
    y_event_multi_temp,
    y_event_bin_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_event_bin_temp
)

print("Train size:", X_train.shape[0])
print("Val size:",   X_val.shape[0])
print("Test size:",  X_test.shape[0])

Train size: 892173
Val size: 191180
Test size: 191180


### Preprocessing pipeline (shared for DeepSurv & DeepHit)

Apply:
- `StandardScaler` to numeric columns
- `OneHotEncoder` (with `handle_unknown="ignore"`) to categorical columns

**Fit only on the training set**, then apply the same transform to val & test.

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Fit on training data only
preprocessor.fit(X_train)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['CONTRACT_CREDIT_LOSS',
                                  'CONTRACT_DATE_OF_LOAN_AGREEMENT',
                                  'CONTRACT_DEPT_SERVICE_TO_INCOME',
                                  'CONTRACT_INCOME',
                                  'CONTRACT_INSTALMENT_AMOUNT_2',
                                  'CONTRACT_INTEREST_PERIOD',
                                  'CONTRACT_INTEREST_RATE', 'CONTRACT_LGD',
                                  'CONTRACT_LOAN_AMOUNT',
                                  'CONTRACT_MATURITY_DATE',
                                  'CONTRACT_RISK_WEIGHTED_ASSETS',
                                  'BORRO...
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['CONTRACT_CREDIT_INTERMEDIARY',
                                  'CONTRACT_CURRENCY',
                                  'CONTRACT_FREQUENCY_TYPE',
                                  'CONTRACT_LOAN_CONTRACT_TYPE',
                                  'CONTRACT_LOAN_TYPE', 'CONTRACT_REFINANCED',
                                  'CONTRACT_TYPE_OF_INTEREST_REPAYMENT',
                                  'BORROWER_CITIZENSHIP', 'BORROWER_COUNTRY',
                                  'BORROWER_COUNTY',
                                  'BORROWER_TYPE_OF_CUSTOMER',
                                  'BORROWER_TYPE_OF_SETTLEMENT'])])

In [20]:
# Transform train / val / test and convert to DataFrames
X_train_enc = preprocessor.transform(X_train)
X_val_enc   = preprocessor.transform(X_val)
X_test_enc  = preprocessor.transform(X_test)

# Get feature names after encoding
feature_names = preprocessor.get_feature_names_out()

# Convert to dense DataFrames
X_train_df = pd.DataFrame(X_train_enc.toarray(), columns=feature_names)
X_val_df   = pd.DataFrame(X_val_enc.toarray(),   columns=feature_names)
X_test_df  = pd.DataFrame(X_test_enc.toarray(),  columns=feature_names)

X_train_df.shape, X_val_df.shape, X_test_df.shape

((892173, 367), (191180, 367), (191180, 367))

In [21]:
# Attach labels and save train / val / test CSVs
# Train
train_out = X_train_df.copy()
train_out["time"]         = y_time_train
train_out["event"]        = y_event_multi_train
train_out["event_binary"] = y_event_bin_train

train_out.to_csv("train_contract_survival.csv", index=False)

# Validation
val_out = X_val_df.copy()
val_out["time"]         = y_time_val
val_out["event"]        = y_event_multi_val
val_out["event_binary"] = y_event_bin_val

val_out.to_csv("val_contract_survival.csv", index=False)

# Test
test_out = X_test_df.copy()
test_out["time"]         = y_time_test
test_out["event"]        = y_event_multi_test
test_out["event_binary"] = y_event_bin_test

test_out.to_csv("test_contract_survival.csv", index=False)

# SUPPORT2

In [65]:
df = pd.read_csv("support2.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9105 entries, 1 to 9105
Data columns (total 47 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       9105 non-null   float64
 1   death     9105 non-null   int64  
 2   sex       9105 non-null   object 
 3   hospdead  9105 non-null   int64  
 4   slos      9105 non-null   int64  
 5   d.time    9105 non-null   int64  
 6   dzgroup   9105 non-null   object 
 7   dzclass   9105 non-null   object 
 8   num.co    9105 non-null   int64  
 9   edu       7471 non-null   float64
 10  income    6123 non-null   object 
 11  scoma     9104 non-null   float64
 12  charges   8933 non-null   float64
 13  totcst    8217 non-null   float64
 14  totmcst   5630 non-null   float64
 15  avtisst   9023 non-null   float64
 16  race      9063 non-null   object 
 17  sps       9104 non-null   float64
 18  aps       9104 non-null   float64
 19  surv2m    9104 non-null   float64
 20  surv6m    9104 non-null   float64
 

In [66]:
drop_cols = [
    "id",
    "hospdead",
    "sfdm2",
    "surv2m",
    "surv6m",
    "prg2m",
    "prg6m",
    "slos",
    "charges",
    "totcst",
    "totmcst",
    "avtisst",
    "dnrday",
    # will also drop after creating time/event:
    "death",
    "d.time",
]
df["time"] = df["d.time"]
df["event"] = df["death"]
existing_to_drop = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing_to_drop)
print("Shape after dropping:", df.shape)

Shape after dropping: (9105, 35)


In [67]:
# Harrell recommended "normal" fill-in values for certain physiologic variables
harrell_defaults = {
    "alb": 3.5,       # serum albumin
    "pafi": 333.3,    # PaO2/FiO2 ratio
    "bili": 1.01,     # bilirubin
    "crea": 1.01,     # creatinine
    "bun": 6.51,      # blood urea nitrogen
    "wblc": 9.0,      # white blood cell count (thousands)
    "urine": 2502.0,  # urine output
}

for col, val in harrell_defaults.items():
    if col in df.columns:
        before = df[col].isna().sum()
        df[col] = df[col].fillna(val)
        after = df[col].isna().sum()
        print(f"{col}: {before - after} missing values filled with {val}")

alb: 3372 missing values filled with 3.5
pafi: 2325 missing values filled with 333.3
bili: 2601 missing values filled with 1.01
crea: 67 missing values filled with 1.01
bun: 4352 missing values filled with 6.51
wblc: 212 missing values filled with 9.0
urine: 4862 missing values filled with 2502.0


In [68]:
# MEDIAN IMPUTATION FOR ALL REMAINING NUMERIC COLUMNS

num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    if df[col].isna().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"{col}: median imputation → {median_val:.3f}")


edu: median imputation → 12.000
scoma: median imputation → 0.000
sps: median imputation → 23.898
aps: median imputation → 34.000
meanbp: median imputation → 77.000
hrt: median imputation → 100.000
resp: median imputation → 24.000
temp: median imputation → 36.695
sod: median imputation → 137.000
ph: median imputation → 7.420
glucose: median imputation → 135.000
adlp: median imputation → 0.000
adls: median imputation → 1.000


In [69]:
# ONE-HOT ENCODE CATEGORICAL COLUMNS

cat_cols = df.select_dtypes(include=["object", "category"]).columns
print("Categorical:", list(cat_cols))

binary_cols = []
multi_cols = []

for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique == 2:
        binary_cols.append(col)
    else:
        multi_cols.append(col)

print("Binary categorical:", binary_cols)
print("Multi-class categorical:", multi_cols)

# Binary encoding
for col in binary_cols:
    df[col] = df[col].astype("category").cat.codes

# One-hot encoding for multi-class
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print("Shape after categorical encoding:", df.shape)

Categorical: ['sex', 'dzgroup', 'dzclass', 'income', 'race', 'ca', 'dnr']
Binary categorical: ['sex']
Multi-class categorical: ['dzgroup', 'dzclass', 'income', 'race', 'ca', 'dnr']
Shape after categorical encoding: (9105, 50)


In [70]:
from sklearn.preprocessing import StandardScaler

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Do NOT scale time/event
for col in ["time", "event"]:
    if col in num_cols:
        num_cols.remove(col)

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [71]:
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

In [73]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

print(train.shape, val.shape, test.shape)

train.to_csv("support2_train.csv", index=False)
val.to_csv("support2_val.csv", index=False)
test.to_csv("support2_test.csv", index=False)

(6373, 50) (1366, 50) (1366, 50)


# Synthetic

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("synthetic_comprisk.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   time        30000 non-null  int64  
 1   label       30000 non-null  int64  
 2   true_time   30000 non-null  int64  
 3   true_label  30000 non-null  int64  
 4   feature1    30000 non-null  float64
 5   feature2    30000 non-null  float64
 6   feature3    30000 non-null  float64
 7   feature4    30000 non-null  float64
 8   feature5    30000 non-null  float64
 9   feature6    30000 non-null  float64
 10  feature7    30000 non-null  float64
 11  feature8    30000 non-null  float64
 12  feature9    30000 non-null  float64
 13  feature10   30000 non-null  float64
 14  feature11   30000 non-null  float64
 15  feature12   30000 non-null  float64
dtypes: float64(12), int64(4)
memory usage: 3.7 MB


In [6]:
# This ID will be used to link data rows to their true labels later
df["id"] = np.arange(len(df))
cols = ["id"] + [c for c in df.columns if c != "id"]
df = df[cols]

# keep true_time and true_label OUTSIDE the train/val/test splits
true_cols = ["id", "true_time", "true_label"]
df_true = df[true_cols].copy()
df_true.to_csv("synthetic_true.csv", index=False)

df = df.drop(columns=["true_time", "true_label"])

In [7]:
train, temp = train_test_split(df, test_size=0.30, random_state=42, shuffle=True)
val, test = train_test_split(temp, test_size=0.50, random_state=42, shuffle=True)

print("Train shape:", train.shape)
print("Val shape:  ", val.shape)
print("Test shape: ", test.shape)

# Quick check: label distribution
print("\nLabel distribution (train):")
print(train["label"].value_counts().sort_index())
print("\nLabel distribution (val):")
print(val["label"].value_counts().sort_index())
print("\nLabel distribution (test):")
print(test["label"].value_counts().sort_index())

Train shape: (21000, 15)
Val shape:   (4500, 15)
Test shape:  (4500, 15)

Label distribution (train):
label
0    10503
1     5318
2     5179
Name: count, dtype: int64

Label distribution (val):
label
0    2264
1    1146
2    1090
Name: count, dtype: int64

Label distribution (test):
label
0    2233
1    1136
2    1131
Name: count, dtype: int64


In [8]:
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)